## OKAERTool and PyNavis initialization

Board: XEM6310 Spartan-6

In [12]:
import sys
import os

# Add the parent directory to the path to import pyOKAERTool (only if the package is not installed)
# sys.path.insert(0, os.path.abspath('..'))
import pyOKAERTool as okt
from pyNAVIS import *

# Define bitfile path
bitfile_path = '../bitfiles/CNAS_okaertool_XEM6310.bit'
# bitfile_path = None  # Set to None if no .bit file is to be used

# Validate the existence of the .bit file
if bitfile_path is None:
    None
elif not os.path.exists(bitfile_path):
    print(f"El archivo .bit no existe en la ruta especificada: {bitfile_path}")
    sys.exit(1)

# Create a new intance of the OkaerTool class and initialize it
okaer = okt.Okaertool(bit_file=bitfile_path)
okaer.init()

# Create a new instance of the PyNAVIS class
settings = MainSettings(num_channels=64, mono_stereo=1, on_off_both=1, address_size=4, ts_tick=0.01, bin_size=10000)

## NAS configuration

In [13]:
import re
import os

config_file_path = '../CFBank_64_20_22000.vhd'

def _tok_to_int(tok):
    tok = tok.strip().rstrip(',').strip()
    if tok.lower().startswith('x"') and tok.endswith('"'):
        return int(tok[2:-1], 16)
    if tok.lower().startswith('0x'):
        return int(tok, 16)
    m = re.match(r'16#([0-9A-Fa-f]+)#', tok)
    if m:
        return int(m.group(1), 16)
    if tok.isdigit():
        return int(tok, 10)
    raise ValueError(f"Unrecognized token: {tok!r}")

def parse_cascade_vhd(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    text = open(path, 'r', encoding='utf-8', errors='ignore').read()

    # Find successive groups of the four parameters in the file order
    pattern = re.compile(
        r'FREQ_DIV\s*=>\s*(?P<f>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_FB\s*=>\s*(?P<fb>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_OUT\s*=>\s*(?P<out>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_BPF\s*=>\s*(?P<bpf>[^,\n;]+)',
        re.IGNORECASE | re.DOTALL
    )

    values = []
    for m in pattern.finditer(text):
        f = _tok_to_int(m.group('f'))
        fb = _tok_to_int(m.group('fb'))
        out = _tok_to_int(m.group('out'))
        bpf = _tok_to_int(m.group('bpf'))
        values.extend([f, fb, out, bpf])

    return values

def reset_and_configure_okaer():
    #Reset the OkaerTool
    okaer.reset_board(mode='internal')

    # Configure the PDM2Spikes (left and right) for both NAS
    register_address = 0x0000
    okaer.logger.info("Configuring PDM2Spikes modules")
    # Left cochlea
    okaer.logger.info("Left cochlea")
    for value in PDM2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)
        register_address += 1
    # Right cochlea
    okaer.logger.info("Right cochlea")
    for value in PDM2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)
        register_address += 1

    register_address = 0x08
    okaer.logger.info("Configuring I2S2Spikes modules")
    # Configure I2S2Spikes modules for both NAS
    for value in I2S2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)

    # Configure the filters for CASCADE NAS
    okaer.logger.info("Configuring filters for Cascade NAS")
    # Left cochlea
    register_address = 0x09
    okaer.logger.info("Left cochlea")
    for value in CASCADE_FILTER_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        register_address += 1
        # # Config only 32 filters
        # if register_address >= 0x09 + 32*4:
        #     break
    # Right cochlea
    register_address = 0x010D
    okaer.logger.info("Right cochlea")
    for value in CASCADE_FILTER_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        register_address += 1
        # # Config only 32 filters
        # if register_address >= 0x010D + 32*4:
        #     break

# Define default parameters for the filters
PDM2Spikes_DEFAULT_parameter = [0x0005, 0x0006, 0x734B, 0x39C8]
I2S2Spikes_DEFAULT_parameter = [0x000F]
CASCADE_FILTER_DEFAULT_parameter = parse_cascade_vhd(config_file_path)

# quick validation / pretty print
filters = len(CASCADE_FILTER_DEFAULT_parameter) // 4
print(f"Parsed {filters} filters ({len(CASCADE_FILTER_DEFAULT_parameter)} values).")
print("CASCADE_FILTER_DEFAULT_parameter = [")
for v in CASCADE_FILTER_DEFAULT_parameter:
    # print as hex literal (4 hex digits minimum)
    width = max(2, (v.bit_length() + 3) // 4)
    print(f"    0x{v:0{width}X},")
print("]")

#reset_and_configure_okaer()

Parsed 65 filters (260 values).
CASCADE_FILTER_DEFAULT_parameter = [
    0x04,
    0x7CB1,
    0x7CB1,
    0x2025,
    0x04,
    0x6F93,
    0x6F93,
    0x2025,
    0x02,
    0x77CE,
    0x77CE,
    0x2025,
    0x02,
    0x6B33,
    0x6B33,
    0x2025,
    0x03,
    0x7FE5,
    0x7FE5,
    0x2025,
    0x03,
    0x7271,
    0x7271,
    0x2025,
    0x03,
    0x6666,
    0x6666,
    0x2025,
    0x04,
    0x7289,
    0x7289,
    0x2025,
    0x02,
    0x7AFB,
    0x7AFB,
    0x2025,
    0x02,
    0x6E0B,
    0x6E0B,
    0x2025,
    0x02,
    0x6277,
    0x6277,
    0x2025,
    0x03,
    0x757A,
    0x757A,
    0x2025,
    0x03,
    0x691E,
    0x691E,
    0x2025,
    0x04,
    0x7593,
    0x7593,
    0x2025,
    0x02,
    0x7E3F,
    0x7E3F,
    0x2025,
    0x02,
    0x70F7,
    0x70F7,
    0x2025,
    0x02,
    0x6514,
    0x6514,
    0x2025,
    0x03,
    0x7898,
    0x7898,
    0x2025,
    0x03,
    0x6BE8,
    0x6BE8,
    0x2025,
    0x04,
    0x78B1,
    0x78B1,
    0x2025,
    0x04,
 

## Experiment

### Audio functions

In [ ]:
import sounddevice as sd
import soundfile as sf
import tkinter as tk
from pathlib import Path
from tkinter import filedialog

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import correlate

def list_output_devices():
    """Prints all available audio output devices and their IDs."""
    print(sd.query_devices())

def select_wav_folder():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    folder_path = filedialog.askdirectory(title="Select a folder containing WAV files")
    return folder_path


def collect_wav_files(folder_path):
    folder = Path(folder_path)
    return sorted(str(path) for path in folder.rglob('*.wav'))


def get_wav_header_info(file_path):
    """Extracts metadata (features) from the WAV header."""
    with sf.SoundFile(file_path) as f:
        info = {
            "samplerate": f.samplerate,
            "channels": f.channels,
            "subtype": f.subtype,      # Bit depth (e.g., PCM_16)
            "format": f.format,        # File format (WAV, FLAC, etc.)
            "frames": f.frames,        # Total number of audio samples
            "duration_sec": f.frames / f.samplerate
        }
    return info


def cross_correlate(recorded_signal, chirp_signal, mode="full", plot=False):
    # FFT-based correlation is usually much faster for long signals.
    correlation = correlate(recorded_signal, chirp_signal, mode=mode, method="fft")

    # Find the index of the maximum correlation value.
    max_corr_index = np.argmax(correlation)

    # Calculate the start index of the chirp in the recorded signal.
    chirp_start_index = max_corr_index - len(chirp_signal) + 1

    if plot:
        plt.plot(correlation)
        plt.show()

    return chirp_start_index


def find_chirp_end(recording_file_path, chirp_file_path, plot=False):
    # Load the recorded signal
    recorded_signal, recorded_sample_rate = sf.read(recording_file_path)
    chirp_signal, chirp_sample_rate = sf.read(chirp_file_path)

    recorded_signal = np.asarray(recorded_signal)
    if recorded_signal.ndim == 1:
        recorded_signal = recorded_signal[:, np.newaxis]

    chirp_signal = np.asarray(chirp_signal)
    if chirp_signal.ndim > 1:
        chirp_signal = chirp_signal[:, 0]

    chirp_signal = chirp_signal * np.max(np.abs(recorded_signal))

    starts = []
    for i in range(recorded_signal.shape[1]):
        starts.append(cross_correlate(recorded_signal[:, i], chirp_signal, mode="full", plot=plot))

    mean_start = int(np.mean(starts))
    chirp_end_index = mean_start + len(chirp_signal)

    if plot:
        fig, ax = plt.subplots(3)
        ax[0].plot(recorded_signal[:, 0])
        ax[1].plot(chirp_signal)
        ax[1].set_xlim(0, recorded_signal.shape[0])
        ax[1].plot(recorded_signal[mean_start:, 0], color="r")
        ax[0].axvline(mean_start, color="green", linestyle="--", label="Detected chirp start")
        ax[0].axvline(chirp_end_index, color="red", linestyle="--", label="Processing start")
        ax[0].legend()
        plt.show()

    return chirp_end_index, chirp_end_index / recorded_sample_rate


def find_interesting_audio_end(recording_file_path, start_index=0, plot=False, plot_output_path=None):
    """
    Detect where interesting audio ends and residual noise remains.

    Algorithm:
    1) Split into ~25 ms frames.
    2) Compute frame RMS.
    3) Smooth RMS with a ~200 ms moving average.
    4) Estimate noise floor from final 1 second.
    5) Use threshold = noise_floor * 3.
    6) Scan backwards for last frame above threshold.
    7) Add ~250 ms safety margin.
    """
    signal, sample_rate = sf.read(recording_file_path)
    signal = np.asarray(signal)

    frame_ms = 25.0
    smoothing_ms = 200.0
    tail_seconds = 1.0
    threshold_multiplier = 3.0
    safety_margin_ms = 750.0  # Updated safety margin to show how spikes disappear after the interesting audio ends

    if signal.ndim == 1:
        mono = signal.astype(np.float64)
    else:
        # Use the channel with the highest global RMS for robust detection.
        channel_rms = np.sqrt(np.mean(signal.astype(np.float64) ** 2, axis=0) + 1e-12)
        mono = signal[:, int(np.argmax(channel_rms))].astype(np.float64)

    n_samples = len(mono)
    if n_samples == 0:
        return 0, 0.0

    frame_length = max(1, int(round((frame_ms / 1000.0) * sample_rate)))
    smoothing_frames = max(1, int(round(smoothing_ms / frame_ms)))
    tail_frames = max(1, int(round((tail_seconds * sample_rate) / frame_length)))
    safety_margin_samples = int(round((safety_margin_ms / 1000.0) * sample_rate))

    n_frames = int(np.ceil(n_samples / frame_length))
    rms = np.empty(n_frames, dtype=np.float64)
    for i in range(n_frames):
        start = i * frame_length
        stop = min(n_samples, start + frame_length)
        frame = mono[start:stop]
        rms[i] = np.sqrt(np.mean(frame**2) + 1e-12)

    if smoothing_frames == 1:
        smoothed_rms = rms
    else:
        kernel = np.ones(smoothing_frames, dtype=np.float64) / smoothing_frames
        smoothed_rms = np.convolve(rms, kernel, mode="same")

    tail = smoothed_rms[-tail_frames:]
    noise_floor = float(np.mean(tail))
    threshold = noise_floor * threshold_multiplier

    start_index = int(np.clip(start_index, 0, n_samples - 1))
    start_frame = max(0, start_index // frame_length)

    last_active_frame = None
    for frame_idx in range(n_frames - 1, start_frame - 1, -1):
        if smoothed_rms[frame_idx] > threshold:
            last_active_frame = frame_idx
            break

    if last_active_frame is None:
        end_sample = start_index
    else:
        frame_end = min(n_samples, (last_active_frame + 1) * frame_length)
        end_sample = min(n_samples, frame_end + safety_margin_samples)

    # Return as sample index in [0, n_samples - 1] for compatibility.
    end_index = int(np.clip(end_sample - 1, 0, n_samples - 1))

    if plot:
        time_axis = (np.arange(n_frames) * frame_length) / sample_rate
        plt.figure()
        plt.plot(time_axis, rms, alpha=0.35, label="Frame RMS")
        plt.plot(time_axis, smoothed_rms, label="Smoothed RMS (~200 ms)")
        plt.axhline(threshold, color="orange", linestyle="--", label="Threshold (3x noise floor)")
        plt.axvline(start_index / sample_rate, color="green", linestyle="--", label="Detected start")
        plt.axvline(end_index / sample_rate, color="red", linestyle="--", label="Detected end")
        plt.xlabel("Time (s)")
        plt.ylabel("RMS")
        plt.legend()
        plt.tight_layout()
        if plot_output_path:
            plt.savefig(plot_output_path)
            plt.close()
        else:
            plt.show()
            
    return end_index, end_index / sample_rate


def play_audio_on_device(data, fs, device_id, block=True):
    """
    Plays audio data through a specific output interface.
    :param data: The audio data to be played
    :param fs: The sample rate of the audio data
    :param device_id: The ID of the device (from list_output_devices)
    """
    sd.default.device = device_id
    sd.play(data, fs)
    if block:
        sd.wait()

def prime_output_device(sample_rate, device_id, warmup_ms=150):
    """
    Opens and warms the output path with silence to reduce first-play latency.
    """
    sd.default.device = device_id
    warmup_ms = int(max(20, warmup_ms))
    warmup_frames = max(1, int((warmup_ms / 1000.0) * sample_rate))

    # Keep channel count consistent with the configured output path.
    try:
        with sd.OutputStream(samplerate=sample_rate, channels=2, dtype='float32'):
            sd.sleep(warmup_ms)
    except Exception:
        # Fallback: send a short silence buffer if stream pre-open is unavailable.
        silence = np.zeros((warmup_frames, 2), dtype=np.float32)
        sd.play(silence, sample_rate, blocking=True)

### Selecting audio interface

In [15]:
output_device = None
if output_device is None:
    list_output_devices()
    output_device = int(input("Please set the output_device variable to the ID of your desired output device: "))

   0 Asignador de sonido Microsoft - Input, MME (2 in, 0 out)
   1 Micrófono (Realtek(R) Audio), MME (2 in, 0 out)
   2 Asignador de sonido Microsoft - Output, MME (0 in, 2 out)
*  3 Realtek HD Audio 2nd output (Re, MME (0 in, 2 out)
   4 Altavoces (Realtek(R) Audio), MME (0 in, 2 out)
   5 Controlador primario de captura de sonido, Windows DirectSound (2 in, 0 out)
   6 Micrófono (Realtek(R) Audio), Windows DirectSound (2 in, 0 out)
   7 Controlador primario de sonido, Windows DirectSound (0 in, 2 out)
   8 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
   9 Altavoces (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  10 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
  11 Altavoces (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
  12 Micrófono (Realtek(R) Audio), Windows WASAPI (2 in, 0 out)
  13 Speakers (Nahimic mirroring Wave Speaker), Windows WDM-KS (0 in, 2 out)
  14 Headphones (Realtek HD Audio 2nd output), W

### Playing audio and monitoring spikes

In [ ]:
import matplotlib.pyplot as plt
import AERzip
import threading
import numpy as np
import time

# Monitor the inputs
INPUTS = ['port_a']  # Monitor only port_a where the CNAS outputs are sent. port_b is not used in this configuration
USB_TRANSFER_LENGTH = 64 * 1024

# Set USB transfer length and number of buffers
okaer.USB_TRANSFER_LENGTH = USB_TRANSFER_LENGTH

# Reset the okaerTool board before monitoring to ensure a clean state
okaer.reset_board(mode='internal')

# Set logger to ERROR level to reduce verbosity during processing
okaer.logger.setLevel(okt.logging.ERROR)

# Set up base directories
base_plot = '../Plots'
base_comp = '../Compressed files'
base_trim = '../Dataset_Trim'

# Fast mode: keep the required board reconfiguration, but skip expensive extras by default
VERBOSE_LOGGING = False
SAVE_PLOTS = True
DAC_WARMUP_MS = 150

def log_info(message, *args):
    if VERBOSE_LOGGING:
        okaer.logger.info(message, *args)

def log_warning(message, *args):
    okaer.logger.warning(message, *args)

# Use fixed chirp reference file from the Auxiliary folder
chirp_file_path = str(Path('../Auxiliary/chirp_series.wav').resolve())

if not Path(chirp_file_path).exists():
    print(f'Chirp file not found: {chirp_file_path}. Aborting.')
else:
    print(f'Using chirp reference: {chirp_file_path}')
    start_time = time.time()

    # Select the folder containing the WAV files to be processed
    wav_folder_path = select_wav_folder()

    # Process each WAV file in the selected folder
    if wav_folder_path:
        wav_files = collect_wav_files(wav_folder_path)
        if not wav_files:
            print(f"No WAV files found in the selected folder: {wav_folder_path}")
        else:
            print(f"Found {len(wav_files)} WAV files in {wav_folder_path}")

            # Create subfolder names based on the selected folder
            wav_root = Path(wav_folder_path).resolve()
            folder_name = wav_root.name
            target_plot = Path(base_plot) / folder_name
            target_comp = Path(base_comp) / folder_name
            target_trim = Path(base_trim) / folder_name
            target_plot.mkdir(parents=True, exist_ok=True)
            target_comp.mkdir(parents=True, exist_ok=True)
            target_trim.mkdir(parents=True, exist_ok=True)

            plot_and_save_threads = []
            plot_lock = threading.Lock()
            worker_errors = []
            primed_sample_rate = None

            for wav_file in wav_files:
                try:
                    wav_path = Path(wav_file)
                    rel_path = wav_path.relative_to(wav_root)
                    file_base = wav_path.stem
                    output_plot_dir = target_plot / rel_path.parent
                    output_comp_dir = target_comp / rel_path.parent
                    output_trim_dir = target_trim / rel_path.parent
                    output_plot_dir.mkdir(parents=True, exist_ok=True)
                    output_comp_dir.mkdir(parents=True, exist_ok=True)
                    output_trim_dir.mkdir(parents=True, exist_ok=True)

                    # Check if output files already exist and skip processing if they do
                    plot_path = output_plot_dir / f'{file_base}_sonogram.png'
                    aer_path = output_comp_dir / f'{file_base}_spikes.aedat'
                    trim_plot_path = output_plot_dir / f'{file_base}_trim_detection.png'
                    trim_audio_path = output_trim_dir / f'{file_base}_trimmed.wav'

                    if plot_path.exists() and aer_path.exists() and trim_plot_path.exists() and trim_audio_path.exists():
                        print(f"Skipping {rel_path}: Output files already exist")
                        continue

                    print(f"Processing audio file: {rel_path}")

                    # Load the WAV file
                    data, fs = sf.read(str(wav_path))

                    # Prime DAC path when sample rate changes to reduce startup delay.
                    if primed_sample_rate != fs:
                        prime_output_device(fs, output_device, warmup_ms=DAC_WARMUP_MS)
                        primed_sample_rate = fs

                    # Detect chirp end (trim start)
                    start_time = time.time()
                    trim_start_index, trim_start_time = find_chirp_end(str(wav_path), chirp_file_path, plot=False)
                    trim_start_index = int(trim_start_index)  # Removing the fractional part to ensure it's an integer index
                    end_time = time.time()
                    print(f"Chirp detection completed in {end_time - start_time:.2f} seconds.")

                    # Detect the end of the target segment (trim end)
                    start_time = time.time()
                    trim_end_index, trim_end_time = find_interesting_audio_end(str(wav_path), start_index=trim_start_index, plot=SAVE_PLOTS, plot_output_path=str(trim_plot_path))
                    end_time = time.time()
                    print(f"Interesting audio detection completed in {end_time - start_time:.2f} seconds.")

                    # Trim the audio data to the target segment
                    data = data[trim_start_index:trim_end_index]

                    # Send it to the cochlea for spike generation and monitor the spikes in a separate thread
                    spikes_result = {}
                    duration = len(data) / fs + 0.1  # Add a small buffer to ensure the entire audio is processed

                    reset_and_configure_okaer()  # Ensure the board is reset and configured before monitoring

                    play_audio_started = threading.Event()
                    def play_audio():
                        play_audio_started.set()  # Signal that audio playback has started
                        #print("Playing audio...")
                        play_audio_on_device(data, fs, output_device, block=True)  # Blocking main thread until playback is complete

                    monitor_started = threading.Event()
                    def monitor_spikes():
                        play_audio_started.wait()  # Wait for the audio playback to start before monitoring

                        monitor_started.set()  # Signal that monitor has started
                        #print("Monitoring spikes...")
                        spikes_result['spikes'] = okaer.monitor(inputs=INPUTS, duration=duration)
                    
                    print("Starting audio playback and spike monitoring...")
                    play_thread = threading.Thread(target=play_audio)
                    monitor_thread = threading.Thread(target=monitor_spikes)

                    play_thread.start()
                    monitor_thread.start()

                    monitor_thread.join()  # Wait for the monitoring to finish before proceeding
                    play_thread.join()  # Wait for the audio playback to finish before proceeding
                    
                    spikes = spikes_result.get('spikes')
                    if not spikes:
                        okaer.logger.error("No spikes were recorded for %s. Skipping.", rel_path)
                        continue

                    print("Spike monitoring completed. Processing results...\n")

                    monitored = spikes[0]
                    addr_len = len(monitored.addresses)
                    ts_len = len(monitored.timestamps)
                    if addr_len != ts_len:
                        okaer.logger.error("Mismatch: %d addresses vs %d timestamps!", addr_len, ts_len)
                        continue

                    timestamps = np.asarray(monitored.timestamps).copy()
                    addresses = np.asarray(monitored.addresses).copy()
                    plot_path_str = str(plot_path)
                    aer_path_str = str(aer_path)
                    rel_path_str = str(rel_path)
                    trim_audio_path_str = str(trim_audio_path)

                    def plot_and_save(addresses_local, timestamps_local, plot_path_local, aer_path_local, trim_audio_path_local, data_local, fs_local, rel_path_local):
                        try:
                            log_info("Spike data OK: %d events.", len(timestamps_local))

                            '''if not np.all(timestamps_local[:-1] <= timestamps_local[1:]):
                                okaer.logger.error("Timestamps are NOT in ascending order!")
                                bad_idx = np.where(timestamps_local[:-1] > timestamps_local[1:])[0][0]
                                okaer.logger.error("First violation at index %d: %d > %d", bad_idx, timestamps_local[bad_idx], timestamps_local[bad_idx + 1])
                            else:
                                log_info("Timestamps are in ascending order")'''

                            '''if np.any(timestamps_local < 0):
                                okaer.logger.error("Found %d negative timestamps!", np.sum(timestamps_local < 0))
                            else:
                                log_info("No negative timestamps")'''

                            # Save event file inside the folder named after the selected folder
                            sf.write(trim_audio_path_local, data_local, fs_local)
                            AERzip.saveCompressedFile(addresses_local, timestamps_local, aer_path_local, overwrite=True)
                        except Exception as worker_exc:
                            worker_errors.append((rel_path_local, str(worker_exc)))

                    plot_and_save_thread = threading.Thread(target=plot_and_save, args=(addresses, timestamps, plot_path_str, aer_path_str, trim_audio_path_str, data, fs, rel_path_str))
                    plot_and_save_thread.start()  # Do not wait for it to finish, let it run in the background while processing the next file
                    plot_and_save_threads.append(plot_and_save_thread)

                    if SAVE_PLOTS:
                        Plots.sonogram(SpikesFile(addresses=addresses, timestamps=timestamps), settings)
                        plt.savefig(plot_path_str)
                        plt.close()
                except Exception as e:
                    okaer.logger.error("Error processing %s: %s", rel_path, e)
                    continue
            for thread in plot_and_save_threads:
                thread.join()
            if worker_errors:
                for rel_path_err, err in worker_errors:
                    okaer.logger.error("Background save failed for %s: %s", rel_path_err, err)
            elapsed_time = time.time() - start_time
            print(f"Finished processing {len(wav_files)} WAV files.")
            print(f"Total processing time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    else:
        print("No folder was selected.")



Using chirp reference: C:/Users/alvco/Desktop/Manchester2026/Auxiliary/chirp_series.wav
Found 460 WAV files in C:/Users/alvco/Desktop/Manchester2026/Dataset/1
Processing audio file: r_0.wav
Chirp detection completed in 0.03 seconds.
Interesting audio detection completed in 0.14 seconds.
Starting audio playback and spike monitoring...
Spike monitoring completed. Processing results...

Processing audio file: r_1.wav
Chirp detection completed in 0.03 seconds.
Interesting audio detection completed in 0.14 seconds.
Starting audio playback and spike monitoring...
Spike monitoring completed. Processing results...

Processing audio file: r_10.wav
Chirp detection completed in 0.03 seconds.
Interesting audio detection completed in 0.14 seconds.
Starting audio playback and spike monitoring...
Spike monitoring completed. Processing results...

Processing audio file: r_100.wav
Chirp detection completed in 0.04 seconds.
Interesting audio detection completed in 0.13 seconds.
Starting audio playback a

KeyboardInterrupt: 